# BDC Satria Data 2026 — EVA-02 Large (Resolusi Native / Akurasi Maksimal, Komputer Kampus)

- **Model:** `eva02_large_patch14_448.mim_m38m_ft_in22k_in1k` — ~305M params, pretraining Merged-38M + MIM (EVA-CLIP teacher), setara ukuran dengan ViT-L/16 kamu.
- **Resolusi:** **448×448** (native EVA-02, akurasi maksimal — sesuai pola setup kamu utk model resolusi-tinggi).

**PENYESUAIAN PENTING dari config yang kamu kasih (WAJIB dibaca):**
1. **`dynamic_img_size=True` ditambahkan di model builder** — EVA-02 pakai RoPE (Rotary Position Embeddings), tanpa flag ini akan langsung error `AssertionError: Input height doesn't match model` (persis error yang kamu alami di EVA-02 Base kemarin, sekarang dicegah dari awal).
2. **`BATCH_SIZE` saya turunkan dari 51 → 8** (dengan `GRAD_ACCUM_STEPS=4`, efektif batch 32). Alasan: config `BATCH_SIZE=51` yang kamu kasih itu hasil tuning untuk Swin-Base V2 (~88M params, res 384). EVA-02 Large ini **~305M params @ 448** — jauh lebih berat (parameter 3.5x lebih banyak, resolusi lebih tinggi, arsitektur ViT-Large lebih boros memory dari CNN/Swin). Kalau saya biarkan `BATCH_SIZE=51`, risiko OOM sangat tinggi meski VRAM 12GB. Silakan naikkan bertahap (16, 24, ...) kalau ternyata VRAM masih longgar.
3. Sisanya (path, `N_FOLDS=3`, `FOLD_VAL_FRAC=0.10`, `NUM_EPOCHS=15`, dst) saya samakan persis seperti yang kamu kasih.

**Konsisten dgn notebook eksplorasi lain:** `StratifiedShuffleSplit` 3-fold 90/10 (mode eksplorasi, bukan OOF/stacking), class weight balanced per fold, macro F1, tracking misclassified → `.xlsx` 2 sheet.

## 1. Setup & Konfigurasi

In [ ]:
from __future__ import annotations
import os
import json
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

try:
    import timm
except ImportError:
    raise ImportError("timm belum terinstall. Jalankan: pip install timm")

In [ ]:
# === PATH DATASET ===
DATA_ROOT = Path(r"C:\Users\MyPC PRO\Downloads\BDC2026\DATA")
TRAIN_DIR = DATA_ROOT / "train"
MODELS_DIR = DATA_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_FOLDERS = ["0_Recyclable", "1_Electronic", "2_Organic"]
CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]
NUM_CLASSES = len(CLASS_NAMES)

# === MODEL ===
MODEL_KEY = "eva02_large"
TIMM_NAME = "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k"
IMG_SIZE = 448           # resolusi native EVA-02 Large (akurasi maksimal)

# === CONFIG TRAINING ===
BATCH_SIZE = 8           # diturunkan dari 51 -- lihat catatan di markdown atas (model 305M @ res 448, jauh lebih berat dari Swin-Base)
GRAD_ACCUM_STEPS = 4     # efektif batch = 32
NUM_EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4
N_FOLDS = 3
FOLD_VAL_FRAC = 0.10
SEED = 42
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_KEY} @ res {IMG_SIZE} | effective batch {BATCH_SIZE * GRAD_ACCUM_STEPS}")

## 2. Index Dataset dari Folder

In [ ]:
def index_dataset(train_dir: Path, class_folders: list[str]):
    records = []
    for label_idx, folder_name in enumerate(class_folders):
        folder = train_dir / folder_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        files = [p for p in folder.iterdir() if p.suffix.lower() in exts]
        for p in files:
            records.append({"path": str(p), "label": label_idx, "class_name": CLASS_NAMES[label_idx]})
    return pd.DataFrame(records)

full_df = index_dataset(TRAIN_DIR, CLASS_FOLDERS)
print(f"Total citra ter-index: {len(full_df)}")
print(full_df["class_name"].value_counts())

## 3. Dataset & Transform (resolusi native + normalisasi asli model)

In [ ]:
class TrashDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        if self.transform:
            img = self.transform(img)
        return img, int(row["label"]), row["path"]


def build_transforms(model, img_size: int, train: bool):
    from torchvision import transforms as T
    data_cfg = timm.data.resolve_model_data_config(model)
    mean, std = data_cfg["mean"], data_cfg["std"]
    resize_to = int(img_size * 1.14)

    if train:
        return T.Compose([
            T.Resize((resize_to, resize_to)),
            T.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ])
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])

## 4. Model Builder (backbone freeze + head-only, `dynamic_img_size=True`)

In [ ]:
def build_model(timm_name: str, num_classes: int):
    print(f"  Loading {timm_name} (download besar, ~1.1GB+ pertama kali)...")
    # dynamic_img_size=True WAJIB utk EVA-02: arsitekturnya pakai RoPE (Rotary Position Embeddings).
    # Tanpa ini, timm akan assert input harus persis 448x448 dan error kalau diubah.
    model = timm.create_model(timm_name, pretrained=True, num_classes=num_classes, dynamic_img_size=True)

    for p in model.parameters():
        p.requires_grad = False

    head_module = None
    for attr in ["head", "classifier", "fc"]:
        if hasattr(model, attr):
            candidate = getattr(model, attr)
            if isinstance(candidate, nn.Module):
                head_module = candidate
                break
    if head_module is None:
        raise AttributeError(f"Head classifier tidak ketemu untuk {timm_name}")

    for p in head_module.parameters():
        p.requires_grad = True

    return model.to(DEVICE)

## 5. Train & Evaluate (Gradient Accumulation)

In [ ]:
scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    optimizer.zero_grad()

    for step, (imgs, labels, _) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * GRAD_ACCUM_STEPS
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    for imgs, labels, paths in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        running_loss += loss.item()

        probs = F.softmax(outputs.float(), dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_paths.extend(paths)
        all_probs.extend(probs.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1, np.array(all_labels), np.array(all_preds), all_paths, np.array(all_probs)

## 6. Cross Validation 3-Fold + Kumpulkan Misclassified

In [ ]:
sss = StratifiedShuffleSplit(n_splits=N_FOLDS, test_size=FOLD_VAL_FRAC, random_state=SEED)

fold_results = []
detail_records = []
eval_counter = defaultdict(int)

for fold, (train_idx, val_idx) in enumerate(sss.split(full_df, full_df["label"]), start=1):
    print(f"\n{'='*64}\n---> FOLD {fold}/{N_FOLDS} <---\n{'='*64}")
    train_df = full_df.iloc[train_idx]
    val_df = full_df.iloc[val_idx]

    weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"  Class weights: {dict(zip(CLASS_NAMES, weights.round(3)))}")

    model = build_model(TIMM_NAME, NUM_CLASSES)
    if fold == 1:
        n_total = sum(p.numel() for p in model.parameters())
        n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Total params: {n_total:,} | Trainable (head): {n_train:,}")

    train_tf = build_transforms(model, IMG_SIZE, train=True)
    val_tf = build_transforms(model, IMG_SIZE, train=False)

    train_loader = DataLoader(TrashDataset(train_df, train_tf),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(TrashDataset(val_df, val_tf),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_f1 = 0.0
    epochs_no_improve = 0
    best_path = MODELS_DIR / f"{MODEL_KEY}_fold{fold}.pth"

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_f1, _, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_f1)

        marker = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            marker = " [*] best"
        else:
            epochs_no_improve += 1

        print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} F1 {tr_f1:.4f} | "
              f"val_loss {val_loss:.4f} F1 {val_f1:.4f}{marker}")

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping.")
            break

    model.load_state_dict(torch.load(best_path, weights_only=True))
    _, final_f1, y_true, y_pred, paths, probs = evaluate(model, val_loader, criterion)
    print(f"  >> Fold {fold} best macro F1: {best_f1:.4f} | re-eval F1: {final_f1:.4f}")

    fold_results.append({"fold": fold, "best_macro_f1": best_f1})

    for i, path in enumerate(paths):
        eval_counter[path] += 1
        if y_true[i] != y_pred[i]:
            p = probs[i]
            detail_records.append({
                "fold": fold, "filepath": path,
                "true_label": CLASS_NAMES[y_true[i]],
                "predicted_label": CLASS_NAMES[y_pred[i]],
                "confidence_pct": round(float(p[y_pred[i]]) * 100, 2),
                "true_label_confidence_pct": round(float(p[y_true[i]]) * 100, 2),
                "prob_recyclable_pct": round(float(p[0]) * 100, 2),
                "prob_electronic_pct": round(float(p[1]) * 100, 2),
                "prob_organic_pct": round(float(p[2]) * 100, 2),
            })

    del model, train_loader, val_loader, optimizer, scheduler
    torch.cuda.empty_cache()

## 7. Ringkasan Hasil 3-Fold

In [ ]:
f1s = [r["best_macro_f1"] for r in fold_results]
print("Ringkasan 3-Fold CV (macro F1):")
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['best_macro_f1']:.4f}")
print(f"\nRata-rata macro F1: {np.mean(f1s):.4f} (+/- {np.std(f1s):.4f})")

print("\nPembanding:")
print("  ViT-B/16 SWAG        : 0.9636")
print("  ConvNeXt V2-Large    : 0.9720")
print("  ViT-L/16 SWAG        : 0.9770  <- terbaik kamu sejauh ini")

pd.DataFrame(fold_results).to_csv(OUTPUT_DIR / "eva02_large_fold_results.csv", index=False)

## 8. Laporan Misclassified (`.xlsx`, 2 sheet)

In [ ]:
detail_df = pd.DataFrame(detail_records, columns=[
    "fold", "filepath", "true_label", "predicted_label",
    "confidence_pct", "true_label_confidence_pct",
    "prob_recyclable_pct", "prob_electronic_pct", "prob_organic_pct",
])

summary_rows = []
if len(detail_df) > 0:
    for path, g in detail_df.groupby("filepath"):
        folds_wrong = sorted(g["fold"].unique().tolist())
        summary_rows.append({
            "filepath": path,
            "true_label": g["true_label"].iloc[0],
            "folds_wrong": ",".join(map(str, folds_wrong)),
            "total_wrong": len(g),
            "total_evaluated": eval_counter[path],
            "error_rate": round(len(g) / eval_counter[path], 3) if eval_counter[path] else None,
            "avg_confidence_pct": round(g["confidence_pct"].mean(), 2),
            "avg_true_label_confidence_pct": round(g["true_label_confidence_pct"].mean(), 2),
            "avg_prob_recyclable_pct": round(g["prob_recyclable_pct"].mean(), 2),
            "avg_prob_electronic_pct": round(g["prob_electronic_pct"].mean(), 2),
            "avg_prob_organic_pct": round(g["prob_organic_pct"].mean(), 2),
        })

summary_df = pd.DataFrame(summary_rows, columns=[
    "filepath", "true_label", "folds_wrong", "total_wrong", "total_evaluated", "error_rate",
    "avg_confidence_pct", "avg_true_label_confidence_pct",
    "avg_prob_recyclable_pct", "avg_prob_electronic_pct", "avg_prob_organic_pct",
])
if len(summary_df) > 0:
    summary_df = summary_df.sort_values(["error_rate", "total_wrong"], ascending=False).reset_index(drop=True)

report_path = OUTPUT_DIR / "misclassified_report_eva02_large.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    detail_df.to_excel(writer, sheet_name="Detail", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"[SAVED] {report_path}")
print(f"  Detail  : {len(detail_df)} baris")
print(f"  Summary : {len(summary_df)} gambar unik")

## Catatan
- OOM? Turunkan `BATCH_SIZE` ke 4 atau 2 (naikkan `GRAD_ACCUM_STEPS` biar efektif batch tetap wajar), atau turunkan `IMG_SIZE` ke 336/224.
- Kalau VRAM ternyata longgar setelah beberapa epoch pertama jalan lancar, boleh naikkan `BATCH_SIZE` bertahap (16, 24, ...) di run berikutnya.
- Untuk stacking nanti: notebook ini masih SSS + hanya simpan yang salah (mode eksplorasi). Perlu retrain dgn `StratifiedKFold` (lihat notebook `OOF_Collection`) kalau hasil EVA-02 Large ini kompetitif dan mau diikutkan stacking.